In [1]:
tabla ='qtiod10'
col_tabla = 'solopefec'
essi='essi_dat_cqx004_etl'
col_essi='sol_fec'
dat= "dat_ceqx004_essi"
col_dat='sol_fec'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [6]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [7]:

fecha= '2023-05-01'

In [8]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

condiag=pd.read_sql_query(f"SELECT id_condia,cod_con FROM dim_condiag", con=connection2)
condiag['cod_con']=condiag['cod_con'].str.strip()

cie10=pd.read_sql_query(f"SELECT id_cie,cod_cie FROM dim_cie10", con=connection2)
cie10['cod_cie']=cie10['cod_cie'].str.strip()

tipdx=pd.read_sql_query(f"SELECT id_tipdx, cod_tdx FROM dim_tipdx", con=connection2)
tipdx['cod_tdx']=tipdx['cod_tdx'].str.strip()

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)

pacsec = pd.read_sql_query(f"SELECT id_pacsec,per_sec FROM dim_pacsec", con=connection2)


In [9]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

#INICIO DEL DAT

In [10]:
base1.shape

(50840, 17)

In [11]:
#Eliminamos las columnas que no se usarán aquí: las descripciones previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['des_cas', 'des_red', 'des_con', 'des_cie', 'des_tip']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)

In [12]:
base1.shape

(50840, 12)

In [13]:
inicioProceso=time.time()

In [14]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [15]:
base1.shape

(50840, 12)

In [16]:
control_a.append(base1.shape[0])

In [17]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(50840, 13)

In [18]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [19]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(50840, 13)

In [20]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [21]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(50840, 13)

In [22]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [23]:
merge.shape

(50840, 13)

In [24]:
condiag=condiag.rename(columns={"cod_con": "con_dia"})
base1['con_dia']=base1['con_dia'].str.strip()
base1["con_dia"]=base1["con_dia"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,condiag,how='left',on='con_dia')
base1=pd.merge(base1,condiag,how='inner',on='con_dia')
base1.shape

(50840, 13)

In [25]:
merge.shape

(50840, 13)

In [26]:
log_falla('id_condia', 'con_dia', True)
log_control('dim_condiag')
base1=base1.drop('con_dia',axis=1)

In [27]:
base1['cod_cie']=base1['cod_cie'].str.strip()
base1["cod_cie"]=base1["cod_cie"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cie10,how='left',on='cod_cie')
base1=pd.merge(base1,cie10,how='inner',on='cod_cie')
base1.shape

(50840, 13)

In [28]:
merge.shape #Puede ser inner

(50840, 13)

In [29]:
log_falla('id_cie', 'cod_cie', True)
log_control('dim_cie10')
base1=base1.drop('cod_cie',axis=1)

In [30]:
tipdx=tipdx.rename(columns={"cod_tdx": "tip_dia", "id_tipdx":"id_tipdia"})
base1['tip_dia']=base1['tip_dia'].str.strip()
base1["tip_dia"]=base1["tip_dia"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,tipdx,how='left',on='tip_dia')
base1=pd.merge(base1,tipdx,how='inner',on='tip_dia')
base1.shape

(50840, 13)

In [31]:
merge.shape #Puede ser inner

(50840, 13)

In [32]:
log_falla('id_tipdia', 'tip_dia', True)
log_control('dim_tipdx')
base1=base1.drop('tip_dia',axis=1)

In [33]:
pacsec=pacsec.rename(columns={"per_sec": "pac_sec"})
pacsec['pac_sec']=pacsec['pac_sec'].astype(str).str.strip()
pacsec["pac_sec"]=pacsec["pac_sec"].str.replace(' ', '', regex=True)
base1['pac_sec']=base1['pac_sec'].astype(str).str.strip()
base1["pac_sec"]=base1["pac_sec"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,pacsec,how='inner',on='pac_sec')
base1=pd.merge(base1,pacsec,how='left',on='pac_sec')
base1.shape

(50840, 13)

In [34]:
merge.shape

(49971, 13)

In [35]:
log_falla('id_pacsec', 'pac_sec', True)
base1=base1.drop('pac_sec',axis=1)
dim.append('dim_pacsec')
control_d.append(base1.shape[0])

In [36]:
base1['sol_fec_temp'] = base1['sol_fec'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"dt_fecha":"sol_fec_temp"})
tiempo["sol_fec_temp"] = tiempo['sol_fec_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='sol_fec_temp')
base1 = pd.merge(base1, tiempo, how='left', on='sol_fec_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_sol","dt_ano":"id_ano_sol","dt_mes":"id_mes_sol",
                            "dt_dia":"id_dia_sol","dt_dia_sem":"id_diasem_sol","dt_sem":"id_sem_sol"})
base1=base1.drop("sol_fec_temp",axis=1)
base1.shape

(50840, 18)

In [37]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50840 entries, 0 to 50839
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sol_num        50840 non-null  float64
 1   sec_num        50840 non-null  float64
 2   ord_dia        50840 non-null  float64
 3   sol_fec        50840 non-null  object 
 4   act_med        50840 non-null  float64
 5   id_oricas      50840 non-null  int64  
 6   id_cas         50840 non-null  int64  
 7   id_red         50840 non-null  int64  
 8   id_condia      50840 non-null  int64  
 9   id_cie         50840 non-null  int64  
 10  id_tipdia      50840 non-null  int64  
 11  id_pacsec      49971 non-null  float64
 12  id_time_sol    50839 non-null  float64
 13  id_mes_sol     50839 non-null  float64
 14  id_dia_sol     50839 non-null  float64
 15  id_diasem_sol  50839 non-null  float64
 16  id_sem_sol     50839 non-null  float64
 17  id_ano_sol     50839 non-null  float64
dtypes: flo

In [53]:
base1["sol_fec"].unique()

array([datetime.date(2029, 7, 10), datetime.date(2023, 5, 3),
       datetime.date(2023, 5, 1), datetime.date(2023, 5, 4),
       datetime.date(2023, 5, 2), datetime.date(2023, 5, 5),
       datetime.date(2023, 5, 6), datetime.date(2023, 5, 7),
       datetime.date(2023, 5, 21), datetime.date(2023, 5, 17),
       datetime.date(2023, 5, 13), datetime.date(2023, 5, 19),
       datetime.date(2023, 5, 8), datetime.date(2023, 5, 23),
       datetime.date(2023, 5, 20), datetime.date(2023, 5, 12),
       datetime.date(2023, 5, 24), datetime.date(2023, 5, 22),
       datetime.date(2023, 5, 9), datetime.date(2023, 5, 25),
       datetime.date(2023, 5, 10), datetime.date(2023, 5, 11),
       datetime.date(2023, 5, 26), datetime.date(2023, 5, 14),
       datetime.date(2023, 5, 15), datetime.date(2023, 5, 16),
       datetime.date(2023, 5, 27), datetime.date(2023, 5, 28),
       datetime.date(2023, 5, 30), datetime.date(2023, 5, 29),
       datetime.date(2023, 5, 18), datetime.date(2023, 5, 31),
 

In [38]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [39]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [40]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [41]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [42]:
base

,id_dia_sol,id_cie,id_condia,id_ano_sol,ord_dia,id_sem_sol,id_time_sol,id_pacsec,id_cas,sol_num,id_oricas,id_tipdia,sol_fec,sec_num,act_med,id_diasem_sol,id_mes_sol,id_red
0,NaN,13763,2,NaN,1.0,NaN,NaN,39641913.0,124,5182.0,1,1,2029-07-10,1.0,6799154.0,NaN,NaN,3
1,3.0,6271,2,2023.0,1.0,1.0,20230503.0,49601660.0,124,37040.0,1,1,2023-05-03,1.0,9304526.0,3.0,5.0,3
2,1.0,4813,2,2023.0,1.0,1.0,20230501.0,51380152.0,124,37002.0,1,1,2023-05-01,1.0,9300188.0,1.0,5.0,3
3,4.0,4813,2,2023.0,1.0,1.0,20230504.0,54467194.0,124,37107.0,1,1,2023-05-04,1.0,9307910.0,4.0,5.0,3
4,2.0,6444,2,2023.0,1.0,1.0,20230502.0,48910916.0,124,37030.0,1,1,2023-05-02,1.0,9299768.0,2.0,5.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50835,26.0,6629,2,2023.0,2.0,1.0,20230626.0,48835480.0,10,12509.0,1,1,2023-06-26,1.0,1430699.0,1.0,6.0,33
50836,27.0,6444,2,2023.0,1.0,1.0,20230627.0,52524473.0,10,12527.0,1,1,2023-06-27,1.0,1429949.0,2.0,6.0,33
50837,21.0,3283,2,2023.0,1.0,1.0,20230621.0,55036953.0,10,12513.0,1,1,2023-06-21,1.0,1423294.0,3.0,6.0,33
50838,23.0,4822,2,2023.0,1.0,1.0,20230623.0,38420196.0,10,12489.0,1,1,2023-06-23,1.0,1426969.0,5.0,6.0,33


In [43]:
base.columns

Index(['id_dia_sol', 'id_cie', 'id_condia', 'id_ano_sol', 'ord_dia',
       'id_sem_sol', 'id_time_sol', 'id_pacsec', 'id_cas', 'sol_num',
       'id_oricas', 'id_tipdia', 'sol_fec', 'sec_num', 'act_med',
       'id_diasem_sol', 'id_mes_sol', 'id_red'],
      dtype='object')

In [44]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

5840

In [45]:
fecha_fin = "2024-12-31"

In [46]:
proceso = "DAT"
cod_proceso = 13

# proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=13", con=connection2)
# proceso = proceso.iloc[0, 0]
# cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=13", con=connection2)
# cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [47]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,13,DAT,dat_ceqx004_essi,2023-06-27,16:00,16:04,dim_oricas,2023-05-01,2024-12-31,50840,50840,0,0,id_oricas
1,13,DAT,dat_ceqx004_essi,2023-06-27,16:00,16:04,dim_cas,2023-05-01,2024-12-31,50840,50840,0,0,id_cas
2,13,DAT,dat_ceqx004_essi,2023-06-27,16:00,16:04,dim_red,2023-05-01,2024-12-31,50840,50840,0,0,id_red
3,13,DAT,dat_ceqx004_essi,2023-06-27,16:00,16:04,dim_condiag,2023-05-01,2024-12-31,50840,50840,0,0,id_condia
4,13,DAT,dat_ceqx004_essi,2023-06-27,16:00,16:04,dim_cie10,2023-05-01,2024-12-31,50840,50840,0,0,id_cie
5,13,DAT,dat_ceqx004_essi,2023-06-27,16:00,16:04,dim_tipdx,2023-05-01,2024-12-31,50840,50840,0,0,id_tipdia
6,13,DAT,dat_ceqx004_essi,2023-06-27,16:00,16:04,dim_pacsec,2023-05-01,2024-12-31,50840,50840,0,0,id_pacsec


In [48]:
tabla_logs.to_sql(name=f'logs', con=connection4, if_exists='append', index=False,chunksize=10000)

7

In [49]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.3 completado satisfactoriamente en 51.227 segundos


In [50]:
connection1.close()
connection2.close()
connection3.close()
connection4.close()

In [51]:
engine1.dispose()
engine2.dispose()
engine3.dispose()
engine4.dispose()